# **Projeto 04 — Segmentação de Tumor Cerebral (Germ Cell Tumor)**

### Este Notebook detalha todas as etapas do projeto de segmentação de imagens de MRI cerebral, com foco no tumor do tipo Germinoma (Germ Cell Tumor).

### O projeto é dividido em 3 etapas:

1. **ET-01** — Análise exploratória do dataset (integridade, consistência, qualidade, distribuição e duplicatas);
2. **ET-02** — Método da literatura (U-Net + ResNet34) e propostas de melhorias;
3. **ET-03** — Implementação do método próprio (U-Net + MobileNetV2 + Tversky Loss), comparação de resultados e conclusões.

**Dataset:** [Brain Tumor 12K MRI Images w Masks, Meta and Bbox](https://www.kaggle.com/datasets/fernando2rad/brain-tumor-12k-mri-images-w-masks-meta-and-bbox)

**Equipe (Squad 04):** Alceu, Alexia, Aline, Ana Karolyne, Caren, Érika, Felipe, Igor e Wallyson.

---
# ETAPA 1 — Análise do Dataset (ATIV-04-ET-01)
---

### 1.1 Instalação das Dependências e Download do Dataset

**Explicação: Instalamos as bibliotecas necessárias para o projeto e baixamos o dataset do Kaggle. O dataset contém mais de 12 mil imagens de MRI cerebral com 38 tipos de tumores, cada imagem acompanhada de sua máscara de segmentação, bounding box e metadados em JSON.**

In [ ]:
# Instalação das bibliotecas necessárias
# - segmentation-models-pytorch: fornece arquiteturas prontas de segmentação (U-Net, etc.)
# - albumentations: augmentation de imagens otimizado para segmentação
# - kaggle: API para download de datasets
# - imagehash: cálculo de hash perceptual para detecção de duplicatas
!pip install -q segmentation-models-pytorch albumentations kaggle imagehash

In [ ]:
# Importação de todas as bibliotecas utilizadas no projeto
import os
import json
import glob
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import imagehash
from PIL import Image
from collections import Counter
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

# Suprimir warnings desnecessários para manter o notebook limpo
warnings.filterwarnings('ignore')

# Fixar seeds para reprodutibilidade dos resultados
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print('Bibliotecas importadas com sucesso!')

In [ ]:
# Verificar se a GPU está disponível
# O treinamento de redes neurais é muito mais rápido em GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Download do dataset via API do Kaggle
# O usuário precisa informar seu username e chave de API
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Username Kaggle: ')
os.environ['KAGGLE_KEY'] = getpass('API Key: ')

!kaggle datasets download -d fernando2rad/brain-tumor-12k-mri-images-w-masks-meta-and-bbox \
    -p /content/dataset --force --quiet

!unzip -qo /content/dataset/*.zip -d /content/dataset
print('Download e extração concluídos!')

### 1.2 Criação do DataFrame com Metadados das Imagens

**Explicação: Carregamos o arquivo DATA.json que contém os metadados de todas as imagens do dataset. A partir dele, criamos um DataFrame do Pandas para facilitar a análise. Cada registro contém informações como: nome do arquivo, tipo do tumor, sequência de MRI (T1, T1C+, T2), dimensões da imagem, localização do tumor e caminhos das máscaras de segmentação.**

In [ ]:
# Localizar o arquivo JSON de metadados no dataset
json_path = glob.glob('/content/dataset/**/DATA.json', recursive=True)[0]
dataset_root = os.path.dirname(json_path)
print(f'Arquivo de metadados: {json_path}')
print(f'Diretório raiz: {dataset_root}')

# Carregar o JSON com os metadados de todas as imagens
with open(json_path, 'r', encoding='utf-8') as f:
    all_data = json.load(f)

print(f'Total de registros no JSON: {len(all_data)}')

# Converter para DataFrame
if isinstance(all_data, list):
    df_all = pd.DataFrame(all_data)
else:
    records = []
    for key, val in all_data.items():
        if isinstance(val, dict):
            val['id'] = key
            records.append(val)
    df_all = pd.DataFrame(records)

print(f'\nColunas do DataFrame: {list(df_all.columns)}')
print(f'Shape: {df_all.shape}')
df_all.head()

### 1.3 Integridade dos Arquivos

**Explicação: Verificamos se todas as imagens referenciadas no JSON realmente existem no diretório do dataset, e se estão no formato esperado (JPG e PNG). Também verificamos o caminho inverso: se existem arquivos no diretório que não estão no JSON. Isso é importante para garantir que não há dados faltando ou inconsistentes.**

In [ ]:
# Função auxiliar para localizar arquivos no dataset
def find_file(filename, search_root='/content/dataset'):
    """Busca um arquivo no dataset, tentando diferentes caminhos."""
    for base in [search_root, dataset_root]:
        full = os.path.join(base, filename)
        if os.path.exists(full):
            return full
    results = glob.glob(f'{search_root}/**/{os.path.basename(filename)}', recursive=True)
    return results[0] if results else None

# Verificar a existência de cada imagem listada no JSON
# Identificar a coluna que contém o caminho da imagem
img_col = None
mask_col = None

for col in df_all.columns:
    cl = col.lower()
    sample_val = str(df_all[col].iloc[0]).lower()
    if 'mask' in cl and ('path' in cl or any(e in sample_val for e in ['.png','.jpg'])):
        mask_col = col
    elif ('file' in cl or 'image' in cl or 'path' == cl) and any(e in sample_val for e in ['.png','.jpg','.jpeg']):
        img_col = col

# Fallback caso não encontre automaticamente
if img_col is None:
    for col in df_all.columns:
        val = str(df_all[col].iloc[0])
        if any(ext in val.lower() for ext in ['.png','.jpg']) and 'mask' not in col.lower():
            img_col = col
            break
if mask_col is None:
    for col in df_all.columns:
        if 'mask' in col.lower():
            mask_col = col
            break

print(f'Coluna de imagem identificada: {img_col}')
print(f'Coluna de máscara identificada: {mask_col}')

# Verificar quantas imagens do JSON existem fisicamente
missing_images = 0
existing_images = 0
for idx in range(len(df_all)):
    path = find_file(str(df_all[img_col].iloc[idx]))
    if path:
        existing_images += 1
    else:
        missing_images += 1

print(f'\n--- INTEGRIDADE DOS ARQUIVOS ---')
print(f'Imagens encontradas: {existing_images}')
print(f'Imagens ausentes: {missing_images}')

In [ ]:
# Verificar os formatos dos arquivos de imagem
# Extraímos a extensão de cada arquivo para ver se há formatos inconsistentes
formats = df_all[img_col].apply(lambda x: str(x).split('.')[-1].lower())
format_counts = formats.value_counts()

print('Formatos de imagem encontrados:')
print(format_counts)

# Gráfico de barras dos formatos
ax = format_counts.plot(kind='bar', title='Quantidade de Imagens por Formato', figsize=(8, 5), color='#4A148C')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.xlabel('Formato')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 1.4 Consistência dos Metadados

**Explicação: Verificamos se há valores ausentes (nulos) nos metadados e se as dimensões das imagens são consistentes. Dimensões fora do esperado podem indicar problemas no dataset que precisam ser tratados antes do treinamento do modelo.**

In [ ]:
# Verificar valores nulos em cada coluna
print('--- VALORES NULOS POR COLUNA ---')
null_counts = df_all.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'Nenhum valor nulo encontrado!')

# Verificar dimensões das imagens
if 'width' in df_all.columns and 'height' in df_all.columns:
    print(f'\n--- DIMENSÕES DAS IMAGENS ---')
    print(df_all[['width', 'height']].describe().loc[['mean', 'std', 'min', 'max']])

    # Gráfico de dispersão das dimensões
    plt.figure(figsize=(8, 6))
    plt.scatter(df_all['width'], df_all['height'], alpha=0.3, s=10, color='#4A148C')
    plt.xlabel('Largura (px)')
    plt.ylabel('Altura (px)')
    plt.title('Distribuição das Dimensões das Imagens')
    plt.tight_layout()
    plt.show()
else:
    print('Colunas de dimensão não encontradas. Verificando manualmente...')
    # Amostrar algumas imagens para verificar dimensões
    sample_dims = []
    for idx in random.sample(range(len(df_all)), min(50, len(df_all))):
        path = find_file(str(df_all[img_col].iloc[idx]))
        if path:
            img = Image.open(path)
            sample_dims.append(img.size)
    dim_counter = Counter(sample_dims)
    print(f'Dimensões encontradas (amostra de {len(sample_dims)} imagens):')
    for dim, count in dim_counter.most_common():
        print(f'  {dim[0]}x{dim[1]}: {count} imagens')

### 1.5 Qualidade das Imagens

**Explicação: Identificamos imagens corrompidas que não podem ser abertas ou processadas. Imagens corrompidas causariam erros durante o treinamento do modelo e precisam ser removidas do dataset.**

In [ ]:
# Verificar imagens corrompidas tentando abrir cada uma
# Fazemos uma amostragem para não demorar muito
count_corrupted = 0
corrupted_list = []
total_checked = 0

# Verificar uma amostra representativa (500 imagens)
sample_indices = random.sample(range(len(df_all)), min(500, len(df_all)))

for idx in sample_indices:
    path = find_file(str(df_all[img_col].iloc[idx]))
    if path:
        total_checked += 1
        try:
            img = Image.open(path)
            img.verify()  # Verifica integridade sem carregar na memória
        except Exception:
            count_corrupted += 1
            corrupted_list.append(path)

print(f'--- QUALIDADE DAS IMAGENS ---')
print(f'Imagens verificadas: {total_checked}')
print(f'Imagens corrompidas: {count_corrupted}')
if corrupted_list:
    print(f'Lista de corrompidas: {corrupted_list}')
else:
    print('Nenhuma imagem corrompida detectada!')

### 1.6 Distribuição das Classes

**Explicação: Analisamos a distribuição dos tipos de tumores no dataset. Isso é importante para identificar desbalanceamento entre as classes, que pode prejudicar o treinamento do modelo. O nosso tumor de interesse — Germinoma — é destacado no gráfico.**

In [ ]:
# Identificar a coluna de classe/tipo de tumor
class_col = None
for col in ['tumor_type', 'class', 'label', 'category']:
    if col in df_all.columns:
        class_col = col
        break

if class_col is None:
    for col in df_all.columns:
        if df_all[col].dtype == 'object' and 2 <= df_all[col].nunique() <= 50:
            class_col = col
            break

print(f'Coluna de classe: {class_col}')
print(f'Total de classes: {df_all[class_col].nunique()}')

# Gráfico de distribuição das classes
class_counts = df_all[class_col].value_counts()

plt.figure(figsize=(14, 6))
colors = ['#7B1FA2' if 'Germinoma' in str(c) or 'Germ' in str(c) else '#B0BEC5' for c in class_counts.index]
ax = class_counts.plot(kind='bar', color=colors)

for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=7)

plt.title('Distribuição das Classes de Tumores no Dataset')
plt.xlabel('Tipo de Tumor')
plt.ylabel('Quantidade de Imagens')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

# Filtrar apenas Germinoma
mask_germ = pd.Series([False] * len(df_all))
for col in df_all.columns:
    if df_all[col].dtype == 'object':
        mask_germ = mask_germ | df_all[col].str.contains(
            'Germinoma|Germ Cell', case=False, na=False
        )

df_germ = df_all[mask_germ].copy().reset_index(drop=True)
print(f'\nImagens de Germinoma: {len(df_germ)}')

In [ ]:
# Distribuição por sequência de MRI dentro do Germinoma
if 'sequence' in df_germ.columns:
    seq_counts = df_germ['sequence'].value_counts()
    print('Distribuição por sequência de MRI (Germinoma):')
    print(seq_counts)

    plt.figure(figsize=(8, 5))
    seq_counts.plot(kind='bar', color='#7B1FA2')
    plt.title('Germinoma — Imagens por Sequência de MRI')
    plt.xlabel('Sequência')
    plt.ylabel('Quantidade')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### 1.7 Verificação de Duplicatas

**Explicação: Utilizamos o hash perceptual (pHash) para identificar imagens visualmente duplicadas no dataset do Germinoma. O pHash gera um "fingerprint" da imagem baseado em seu conteúdo visual, permitindo detectar duplicatas mesmo que os arquivos tenham nomes ou tamanhos diferentes. Imagens duplicadas podem enviesar o treinamento do modelo.**

In [ ]:
# Calcular o hash perceptual de cada imagem do Germinoma
hashes = []
valid_paths = []

for idx in range(len(df_germ)):
    path = find_file(str(df_germ[img_col].iloc[idx]))
    if path:
        try:
            img = Image.open(path)
            h = imagehash.phash(img)
            hashes.append(str(h))
            valid_paths.append(path)
        except Exception:
            hashes.append(None)
            valid_paths.append(None)
    else:
        hashes.append(None)
        valid_paths.append(None)

df_germ['image_hash'] = hashes
df_germ['resolved_path'] = valid_paths

# Contar duplicatas
df_valid = df_germ.dropna(subset=['image_hash'])
n_duplicates = df_valid['image_hash'].duplicated().sum()

print(f'--- DUPLICATAS ---')
print(f'Total de imagens analisadas: {len(df_valid)}')
print(f'Imagens duplicadas encontradas: {n_duplicates}')

if n_duplicates > 0:
    dup_hashes = df_valid[df_valid['image_hash'].duplicated(keep=False)]
    print(f'\nGrupos de duplicatas:')
    for h in dup_hashes['image_hash'].unique():
        group = dup_hashes[dup_hashes['image_hash'] == h]
        print(f'  Hash {h}: {len(group)} imagens')
else:
    print('Nenhuma duplicata encontrada no conjunto de Germinoma!')

### 1.8 Visualização de Amostras do Germinoma

**Explicação: Visualizamos algumas amostras de imagens de MRI do Germinoma junto com suas máscaras de segmentação correspondentes. A máscara indica a região exata do tumor na imagem, sendo a "resposta correta" que o modelo precisa aprender a prever.**

In [ ]:
# Visualizar 6 amostras de Germinoma com suas máscaras
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
samples = df_germ.sample(n=min(6, len(df_germ)), random_state=42)

for i, (_, entry) in enumerate(samples.iterrows()):
    row = i // 2
    col_start = (i % 2) * 2

    # Carregar e exibir a imagem original
    img_path = find_file(str(entry[img_col]))
    if img_path:
        img = np.array(Image.open(img_path).convert('RGB'))
        axes[row][col_start].imshow(img)
        axes[row][col_start].set_title('Imagem', fontsize=10)
    axes[row][col_start].axis('off')

    # Carregar e exibir a máscara de segmentação
    if mask_col:
        mask_path = find_file(str(entry[mask_col]))
        if mask_path:
            mask = np.array(Image.open(mask_path).convert('L'))
            axes[row][col_start+1].imshow(mask, cmap='gray')
            axes[row][col_start+1].set_title('Máscara', fontsize=10)
    axes[row][col_start+1].axis('off')

plt.suptitle('Amostras de Germinoma (Germ Cell Tumor)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# ETAPA 2 — Método da Literatura (ATIV-04-ET-02)
---

### 2.1 Preparação dos Dados para Treinamento

**Explicação: Criamos um pipeline de dados para o treinamento do modelo de segmentação. Isso envolve:**
- **Dataset customizado**: classe que carrega pares imagem/máscara e aplica transformações;
- **Data Augmentation**: técnicas de aumento de dados (flip, rotação, brilho) para melhorar a generalização;
- **Normalização**: valores dos pixels normalizados com média e desvio padrão do ImageNet;
- **Split**: divisão dos dados em treino (70%), validação (15%) e teste (15%).

In [ ]:
# ============================================================
# HIPERPARÂMETROS
# ============================================================
# IMG_SIZE: Tamanho para o qual todas as imagens serão redimensionadas.
#   Valor menor = treino mais rápido, porém perde detalhes.
# BATCH_SIZE: Número de imagens processadas por vez pela GPU.
#   Valores maiores exigem mais memória de GPU.
# EPOCHS: Número de vezes que o modelo percorre todos os dados de treino.
# LR: Taxa de aprendizado — controla o tamanho dos ajustes nos pesos.
# ============================================================
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-4

In [ ]:
# ============================================================
# DATASET CUSTOMIZADO
# ============================================================
# Esta classe herda de torch.utils.data.Dataset e é responsável por:
# 1. Resolver os caminhos das imagens e máscaras
# 2. Carregar cada par (imagem, máscara) quando solicitado
# 3. Aplicar as transformações de augmentation
# 4. Binarizar a máscara (pixels > 127 = tumor, resto = fundo)
# ============================================================
class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, img_col, mask_col, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_col = img_col
        self.mask_col = mask_col
        self.transform = transform

        # Resolver caminhos e filtrar pares válidos
        self.img_paths = []
        self.mask_paths = []
        valid = []
        for idx in range(len(self.data)):
            ip = find_file(str(self.data[self.img_col].iloc[idx]))
            mp = find_file(str(self.data[self.mask_col].iloc[idx]))
            if ip and mp:
                self.img_paths.append(ip)
                self.mask_paths.append(mp)
                valid.append(idx)
        self.data = self.data.iloc[valid].reset_index(drop=True)
        print(f'  {len(self.img_paths)} pares imagem/máscara válidos')

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Carregar imagem em RGB e máscara em escala de cinza
        image = np.array(Image.open(self.img_paths[idx]).convert('RGB'))
        mask = np.array(Image.open(self.mask_paths[idx]).convert('L'))

        # Binarizar a máscara: pixel > 127 = 1 (tumor), senão = 0 (fundo)
        mask = (mask > 127).astype(np.float32)

        # Aplicar transformações (augmentation)
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            mask = aug['mask']

        # Garantir que a máscara tem dimensão de canal (1, H, W)
        if isinstance(mask, torch.Tensor) and mask.dim() == 2:
            mask = mask.unsqueeze(0)
        elif isinstance(mask, np.ndarray) and mask.ndim == 2:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

In [ ]:
# ============================================================
# PIPELINE DE TRANSFORMAÇÕES (AUGMENTATION)
# ============================================================
# Treino: aplicamos augmentations para artificialmente aumentar
#   a variedade dos dados e melhorar a generalização do modelo.
# Validação/Teste: apenas resize e normalização, sem augmentation.
# ============================================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),          # Espelhar horizontalmente
    A.VerticalFlip(p=0.3),             # Espelhar verticalmente
    A.RandomRotate90(p=0.3),           # Rotação de 90 graus
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.4),
    A.RandomBrightnessContrast(p=0.3), # Variar brilho e contraste
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# ============================================================
# SPLIT DOS DADOS: 70% treino / 15% validação / 15% teste
# ============================================================
train_df, temp_df = train_test_split(df_germ, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f'Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}')

# Criar os datasets e dataloaders
train_dataset = BrainTumorDataset(train_df, img_col, mask_col, train_transform)
val_dataset = BrainTumorDataset(val_df, img_col, mask_col, val_transform)
test_dataset = BrainTumorDataset(test_df, img_col, mask_col, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

### 2.2 Métricas de Avaliação

**Explicação: Definimos as métricas para avaliar a qualidade da segmentação:**
- **Dice Score**: mede a sobreposição entre predição e máscara real (0 = nenhuma, 1 = perfeita);
- **IoU (Intersection over Union)**: outra medida de sobreposição, mais estrita que o Dice;
- **Pixel Accuracy**: porcentagem de pixels classificados corretamente.

In [ ]:
# ============================================================
# FUNÇÕES DE MÉTRICAS
# ============================================================
def dice_score(preds, targets, smooth=1e-6):
    """Dice Score: 2 * interseção / (soma das áreas).
    Quanto mais próximo de 1, melhor a segmentação."""
    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)
    intersection = (preds_flat * targets_flat).sum()
    return (2. * intersection + smooth) / (preds_flat.sum() + targets_flat.sum() + smooth)

def iou_score(preds, targets, smooth=1e-6):
    """IoU: interseção / união. Mais estrito que o Dice."""
    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)
    intersection = (preds_flat * targets_flat).sum()
    union = preds_flat.sum() + targets_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def pixel_accuracy(preds, targets):
    """Porcentagem de pixels classificados corretamente."""
    correct = (preds == targets).sum().float()
    total = targets.numel()
    return correct / total

In [ ]:
# ============================================================
# FUNÇÕES DE TREINO E AVALIAÇÃO
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Treina o modelo por uma época completa."""
    model.train()  # Ativa modo de treinamento (dropout, batchnorm, etc.)
    total_loss, total_dice = 0, 0

    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        # Forward pass: modelo gera a predição
        outputs = model(images)

        # Calcular a perda
        loss = criterion(outputs, masks)

        # Backward pass: calcular os gradientes
        optimizer.zero_grad()
        loss.backward()

        # Atualizar os pesos do modelo
        optimizer.step()

        # Calcular métricas
        preds = (torch.sigmoid(outputs) > 0.5).float()
        total_loss += loss.item()
        total_dice += dice_score(preds, masks).item()

    return total_loss / len(loader), total_dice / len(loader)

@torch.no_grad()  # Desativa cálculo de gradientes (economia de memória)
def evaluate(model, loader, criterion, device):
    """Avalia o modelo no conjunto de validação/teste."""
    model.eval()  # Ativa modo de avaliação
    total_loss, total_dice, total_iou, total_acc = 0, 0, 0, 0

    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        preds = (torch.sigmoid(outputs) > 0.5).float()

        total_loss += loss.item()
        total_dice += dice_score(preds, masks).item()
        total_iou += iou_score(preds, masks).item()
        total_acc += pixel_accuracy(preds, masks).item()

    n = len(loader)
    return total_loss/n, total_dice/n, total_iou/n, total_acc/n

### 2.3 Modelo da Literatura: U-Net + ResNet34

**Explicação: A U-Net (Ronneberger et al., 2015) é a arquitetura mais consolidada para segmentação de imagens médicas. Ela funciona com um encoder (que extrai features da imagem) e um decoder (que reconstrói a máscara de segmentação). As skip connections conectam encoder e decoder para preservar detalhes espaciais.**

**Configuração do método da literatura:**
- **Encoder**: ResNet34 pré-treinado no ImageNet
- **Loss**: Dice Loss + Binary Cross-Entropy (combinação mais comum na literatura)
- **Otimizador**: Adam com learning rate 1e-4
- **Scheduler**: ReduceLROnPlateau (reduz lr quando a métrica para de melhorar)

In [ ]:
# ============================================================
# MODELO DA LITERATURA: U-Net + ResNet34
# ============================================================
# Utilizamos a biblioteca segmentation-models-pytorch que já fornece
# a arquitetura U-Net com diversos encoders pré-treinados.
# O ResNet34 é uma escolha popular por ser eficiente e ter bom desempenho.
# ============================================================
model_lit = smp.Unet(
    encoder_name='resnet34',        # Encoder pré-treinado
    encoder_weights='imagenet',     # Pesos do ImageNet (transfer learning)
    in_channels=3,                  # Imagem RGB = 3 canais
    classes=1,                      # Saída binária (tumor ou fundo)
    activation=None                 # Sigmoid será aplicado depois
).to(device)

# Função de perda combinada: Dice Loss + BCE
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
bce_loss_fn = nn.BCEWithLogitsLoss()

def criterion_lit(pred, target):
    """Combina Dice Loss e BCE com peso igual (0.5 cada)."""
    return 0.5 * dice_loss_fn(pred, target) + 0.5 * bce_loss_fn(pred, target)

# Otimizador Adam e scheduler ReduceLROnPlateau
optimizer_lit = optim.Adam(model_lit.parameters(), lr=LR)
scheduler_lit = optim.lr_scheduler.ReduceLROnPlateau(optimizer_lit, mode='max', patience=5, factor=0.5)

# Exibir informações do modelo
n_params = sum(p.numel() for p in model_lit.parameters())
print(f'Modelo: U-Net + ResNet34')
print(f'Total de parâmetros: {n_params:,}')

In [ ]:
# ============================================================
# LOOP DE TREINAMENTO — MÉTODO DA LITERATURA
# ============================================================
hist_lit = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'val_acc': []}
best_dice_lit = 0

for epoch in range(EPOCHS):
    # Treinar por uma época
    train_loss, train_dice = train_one_epoch(model_lit, train_loader, criterion_lit, optimizer_lit, device)

    # Avaliar na validação
    val_loss, val_dice, val_iou, val_acc = evaluate(model_lit, val_loader, criterion_lit, device)

    # Ajustar learning rate com base no Dice Score
    scheduler_lit.step(val_dice)

    # Salvar histórico
    hist_lit['train_loss'].append(train_loss)
    hist_lit['train_dice'].append(train_dice)
    hist_lit['val_loss'].append(val_loss)
    hist_lit['val_dice'].append(val_dice)
    hist_lit['val_iou'].append(val_iou)
    hist_lit['val_acc'].append(val_acc)

    # Salvar o melhor modelo
    if val_dice > best_dice_lit:
        best_dice_lit = val_dice
        torch.save(model_lit.state_dict(), '/content/best_model_lit.pth')

    # Log a cada 5 épocas
    if (epoch + 1) % 5 == 0:
        print(f'Epoca {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f} - '
              f'Train Dice: {train_dice:.4f} - Val Dice: {val_dice:.4f} - Val IoU: {val_iou:.4f}')

print(f'\nMelhor Val Dice (Literatura): {best_dice_lit:.4f}')

In [ ]:
# Avaliar no conjunto de teste
model_lit.load_state_dict(torch.load('/content/best_model_lit.pth'))
test_loss_lit, test_dice_lit, test_iou_lit, test_acc_lit = evaluate(model_lit, test_loader, criterion_lit, device)

print(f'=== RESULTADOS NO TESTE — MÉTODO DA LITERATURA ===')
print(f'  Loss:           {test_loss_lit:.4f}')
print(f'  Dice Score:     {test_dice_lit:.4f}')
print(f'  IoU:            {test_iou_lit:.4f}')
print(f'  Pixel Accuracy: {test_acc_lit:.4f}')

In [ ]:
# Visualização das curvas de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, hist_lit['train_loss'], label='Treino')
axes[0].plot(epochs_range, hist_lit['val_loss'], label='Validação')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, hist_lit['train_dice'], label='Treino Dice')
axes[1].plot(epochs_range, hist_lit['val_dice'], label='Val Dice')
axes[1].plot(epochs_range, hist_lit['val_iou'], label='Val IoU', linestyle='--')
axes[1].set_title('Métricas')
axes[1].set_xlabel('Época')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'U-Net + ResNet34 — Curvas de Treinamento (Melhor Dice: {best_dice_lit:.4f})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predições do modelo da literatura
model_lit.eval()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
indices = random.sample(range(len(test_dataset)), min(4, len(test_dataset)))

for i, idx in enumerate(indices):
    image, mask = test_dataset[idx]
    with torch.no_grad():
        pred = torch.sigmoid(model_lit(image.unsqueeze(0).to(device)))
    pred = (pred > 0.5).float().cpu().squeeze().numpy()

    # Desnormalizar a imagem para visualização
    img_np = image.numpy().transpose(1, 2, 0)
    img_np = (img_np * std + mean).clip(0, 1)
    mask_np = mask.squeeze().numpy()

    d = dice_score(torch.from_numpy(pred).unsqueeze(0), mask.squeeze().unsqueeze(0)).item()

    axes[i][0].imshow(img_np)
    axes[i][0].set_title('Imagem Original')
    axes[i][0].axis('off')

    axes[i][1].imshow(mask_np, cmap='gray')
    axes[i][1].set_title('Máscara Real')
    axes[i][1].axis('off')

    axes[i][2].imshow(pred, cmap='gray')
    axes[i][2].set_title(f'Predição (Dice: {d:.3f})')
    axes[i][2].axis('off')

plt.suptitle('Predições — U-Net + ResNet34 (Literatura)', fontweight='bold')
plt.tight_layout()
plt.show()

### 2.4 Propostas de Melhorias (Métodos Próprios)

Com base nos resultados do método da literatura, propomos 3 melhorias fundamentadas:

**1. Otimização de Hiperparâmetros:**
- Trocar o scheduler ReduceLROnPlateau por **CosineAnnealingLR**, que faz um decaimento suave da taxa de aprendizado;
- Usar **Tversky Loss** (alpha=0.7, beta=0.3) no lugar da Dice+BCE. A Tversky penaliza mais os falsos negativos, o que é crucial em diagnóstico médico — não detectar um tumor é mais grave que um falso alarme.

**2. Arquitetura Mais Leve:**
- Substituir o encoder **ResNet34** (~24M parâmetros) por **MobileNetV2** (~3.4M parâmetros), que é 7x mais leve e otimizado para eficiência computacional.

**3. Redução do Número de Imagens:**
- Treinar com apenas **50% das imagens de treino** e compensar com augmentation mais agressiva para testar a robustez do modelo.

---
# ETAPA 3 — Método Próprio e Comparação (ATIV-04-ET-03)
---

### 3.1 Método Próprio: U-Net + MobileNetV2 + Tversky Loss

**Explicação: Implementamos as melhorias propostas na ET-02. O MobileNetV2 usa blocos de convolução mais eficientes (depthwise separable convolutions) que reduzem o custo computacional sem perder representatividade. A Tversky Loss é uma generalização da Dice Loss que permite controlar separadamente o peso dos falsos positivos (FP) e falsos negativos (FN).**

In [ ]:
# ============================================================
# TVERSKY LOSS — IMPLEMENTAÇÃO
# ============================================================
# alpha controla o peso dos falsos positivos
# beta controla o peso dos falsos negativos
# Com alpha=0.3 e beta=0.7, penalizamos mais os FN
# (não detectar o tumor é pior que um falso alarme)
# ============================================================
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha = alpha  # Peso dos falsos positivos
        self.beta = beta    # Peso dos falsos negativos
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)  # Converter logits para probabilidades
        preds_flat = preds.view(-1)
        targets_flat = targets.view(-1)

        # Calcular os componentes
        TP = (preds_flat * targets_flat).sum()              # Verdadeiros positivos
        FP = ((1 - targets_flat) * preds_flat).sum()        # Falsos positivos
        FN = (targets_flat * (1 - preds_flat)).sum()        # Falsos negativos

        # Índice de Tversky
        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        return 1 - tversky  # Loss = 1 - índice

In [ ]:
# ============================================================
# MODELO PRÓPRIO: U-Net + MobileNetV2
# ============================================================
# MobileNetV2 tem ~3.4M parâmetros vs ~24M do ResNet34
# Usa depthwise separable convolutions = mais eficiente
# ============================================================
model_own = smp.Unet(
    encoder_name='mobilenet_v2',     # Encoder mais leve
    encoder_weights='imagenet',      # Transfer learning
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# Loss: Tversky + BCE (combinação própria)
tversky_loss_fn = TverskyLoss(alpha=0.3, beta=0.7)
bce_loss_fn2 = nn.BCEWithLogitsLoss()

def criterion_own(pred, target):
    """Combina Tversky Loss e BCE."""
    return 0.5 * tversky_loss_fn(pred, target) + 0.5 * bce_loss_fn2(pred, target)

# Otimizador e CosineAnnealingLR (decaimento suave)
optimizer_own = optim.Adam(model_own.parameters(), lr=LR)
scheduler_own = optim.lr_scheduler.CosineAnnealingLR(optimizer_own, T_max=EPOCHS, eta_min=1e-6)

n_params_own = sum(p.numel() for p in model_own.parameters())
print(f'Modelo: U-Net + MobileNetV2')
print(f'Total de parâmetros: {n_params_own:,}')
print(f'Redução: {(1 - n_params_own/n_params)*100:.1f}% menos parâmetros que ResNet34')

In [ ]:
# ============================================================
# LOOP DE TREINAMENTO — MÉTODO PRÓPRIO (100% dos dados)
# ============================================================
hist_own = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'val_acc': []}
best_dice_own = 0

for epoch in range(EPOCHS):
    train_loss, train_dice = train_one_epoch(model_own, train_loader, criterion_own, optimizer_own, device)
    val_loss, val_dice, val_iou, val_acc = evaluate(model_own, val_loader, criterion_own, device)
    scheduler_own.step()

    hist_own['train_loss'].append(train_loss)
    hist_own['train_dice'].append(train_dice)
    hist_own['val_loss'].append(val_loss)
    hist_own['val_dice'].append(val_dice)
    hist_own['val_iou'].append(val_iou)
    hist_own['val_acc'].append(val_acc)

    if val_dice > best_dice_own:
        best_dice_own = val_dice
        torch.save(model_own.state_dict(), '/content/best_model_own.pth')

    if (epoch + 1) % 5 == 0:
        print(f'Epoca {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f} - '
              f'Train Dice: {train_dice:.4f} - Val Dice: {val_dice:.4f} - Val IoU: {val_iou:.4f}')

print(f'\nMelhor Val Dice (Próprio): {best_dice_own:.4f}')

In [ ]:
# Avaliar no teste
model_own.load_state_dict(torch.load('/content/best_model_own.pth'))
test_loss_own, test_dice_own, test_iou_own, test_acc_own = evaluate(model_own, test_loader, criterion_own, device)

print(f'=== RESULTADOS NO TESTE — MÉTODO PRÓPRIO ===')
print(f'  Loss:           {test_loss_own:.4f}')
print(f'  Dice Score:     {test_dice_own:.4f}')
print(f'  IoU:            {test_iou_own:.4f}')
print(f'  Pixel Accuracy: {test_acc_own:.4f}')

### 3.2 Experimento: Redução de Dados (50% do Treino)

**Explicação: Treinamos o mesmo modelo próprio (MobileNetV2 + Tversky) utilizando apenas 50% das imagens de treino, com augmentation mais agressiva para compensar a redução. O objetivo é avaliar se o modelo mantém um desempenho razoável com menos dados disponíveis.**

In [ ]:
# ============================================================
# EXPERIMENTO DE REDUÇÃO — 50% DOS DADOS DE TREINO
# ============================================================
# Augmentation mais agressiva para compensar a redução
train_transform_agg = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.15, rotate_limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3),  # Deformação elástica
    A.GridDistortion(p=0.3),                        # Distorção em grade
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Usar apenas 50% dos dados de treino
train_df_reduced, _ = train_test_split(train_df, test_size=0.5, random_state=42)
print(f'Treino reduzido: {len(train_df_reduced)} imagens (50%)')

train_dataset_red = BrainTumorDataset(train_df_reduced, img_col, mask_col, train_transform_agg)
train_loader_red = DataLoader(train_dataset_red, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
# Modelo para o experimento reduzido
model_red = smp.Unet(
    encoder_name='mobilenet_v2',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None
).to(device)

optimizer_red = optim.Adam(model_red.parameters(), lr=LR)
scheduler_red = optim.lr_scheduler.CosineAnnealingLR(optimizer_red, T_max=EPOCHS, eta_min=1e-6)

# Treinar com 50% dos dados
hist_red = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'val_acc': []}
best_dice_red = 0

for epoch in range(EPOCHS):
    train_loss, train_dice = train_one_epoch(model_red, train_loader_red, criterion_own, optimizer_red, device)
    val_loss, val_dice, val_iou, val_acc = evaluate(model_red, val_loader, criterion_own, device)
    scheduler_red.step()

    hist_red['train_loss'].append(train_loss)
    hist_red['train_dice'].append(train_dice)
    hist_red['val_loss'].append(val_loss)
    hist_red['val_dice'].append(val_dice)
    hist_red['val_iou'].append(val_iou)
    hist_red['val_acc'].append(val_acc)

    if val_dice > best_dice_red:
        best_dice_red = val_dice
        torch.save(model_red.state_dict(), '/content/best_model_red.pth')

    if (epoch + 1) % 5 == 0:
        print(f'Epoca {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f} - '
              f'Train Dice: {train_dice:.4f} - Val Dice: {val_dice:.4f} - Val IoU: {val_iou:.4f}')

print(f'\nMelhor Val Dice (Reduzido): {best_dice_red:.4f}')

In [ ]:
# Avaliar no teste
model_red.load_state_dict(torch.load('/content/best_model_red.pth'))
test_loss_red, test_dice_red, test_iou_red, test_acc_red = evaluate(model_red, test_loader, criterion_own, device)

print(f'=== RESULTADOS NO TESTE — MÉTODO PRÓPRIO (50% dados) ===')
print(f'  Loss:           {test_loss_red:.4f}')
print(f'  Dice Score:     {test_dice_red:.4f}')
print(f'  IoU:            {test_iou_red:.4f}')
print(f'  Pixel Accuracy: {test_acc_red:.4f}')

### 3.3 Comparação dos Resultados

**Explicação: Comparamos os três métodos em termos de métricas de segmentação (Dice, IoU, Accuracy) e eficiência (número de parâmetros). A tabela e os gráficos a seguir resumem os resultados.**

In [ ]:
# ============================================================
# TABELA COMPARATIVA DE RESULTADOS
# ============================================================
results = pd.DataFrame({
    'Método': ['U-Net + ResNet34 (Literatura)', 'U-Net + MobileNetV2 (Próprio)', 'MobileNetV2 + 50% dados (Próprio)'],
    'Parâmetros': [f'{n_params:,}', f'{n_params_own:,}', f'{n_params_own:,}'],
    'Dice Score': [f'{test_dice_lit:.4f}', f'{test_dice_own:.4f}', f'{test_dice_red:.4f}'],
    'IoU': [f'{test_iou_lit:.4f}', f'{test_iou_own:.4f}', f'{test_iou_red:.4f}'],
    'Pixel Acc': [f'{test_acc_lit:.4f}', f'{test_acc_own:.4f}', f'{test_acc_red:.4f}'],
    'Loss': [f'{test_loss_lit:.4f}', f'{test_loss_own:.4f}', f'{test_loss_red:.4f}'],
})

print('\n' + '='*80)
print('TABELA COMPARATIVA — RESULTADOS NO TESTE')
print('='*80)
print(results.to_string(index=False))
print('='*80)

In [ ]:
# Gráfico comparativo das métricas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = ['Literatura\n(ResNet34)', 'Próprio\n(MobileNetV2)', 'Próprio\n(50% dados)']
colors = ['#4A148C', '#00695C', '#E65100']

# Dice Score
dice_values = [test_dice_lit, test_dice_own, test_dice_red]
bars = axes[0].bar(methods, dice_values, color=colors)
axes[0].set_title('Dice Score', fontweight='bold')
axes[0].set_ylim(0, 1)
for bar, val in zip(bars, dice_values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f'{val:.4f}', ha='center', fontsize=11)

# IoU
iou_values = [test_iou_lit, test_iou_own, test_iou_red]
bars = axes[1].bar(methods, iou_values, color=colors)
axes[1].set_title('IoU', fontweight='bold')
axes[1].set_ylim(0, 1)
for bar, val in zip(bars, iou_values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f'{val:.4f}', ha='center', fontsize=11)

# Pixel Accuracy
acc_values = [test_acc_lit, test_acc_own, test_acc_red]
bars = axes[2].bar(methods, acc_values, color=colors)
axes[2].set_title('Pixel Accuracy', fontweight='bold')
axes[2].set_ylim(0, 1)
for bar, val in zip(bars, acc_values):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f'{val:.4f}', ha='center', fontsize=11)

plt.suptitle('Comparação dos Métodos — Métricas no Teste', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparação das curvas de treinamento (Dice Score)
plt.figure(figsize=(12, 5))
plt.plot(range(1, EPOCHS+1), hist_lit['val_dice'], label='Literatura (ResNet34)', color='#4A148C', linewidth=2)
plt.plot(range(1, EPOCHS+1), hist_own['val_dice'], label='Próprio (MobileNetV2)', color='#00695C', linewidth=2)
plt.plot(range(1, EPOCHS+1), hist_red['val_dice'], label='Próprio (50% dados)', color='#E65100', linewidth=2, linestyle='--')
plt.xlabel('Época')
plt.ylabel('Val Dice Score')
plt.title('Comparação — Dice Score na Validação por Época', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predições do método próprio (comparação visual)
model_own.eval()
model_lit.eval()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
cols_titles = ['Imagem Original', 'Máscara Real', 'Predição (Literatura)', 'Predição (Próprio)']

indices = random.sample(range(len(test_dataset)), min(3, len(test_dataset)))

for i, idx in enumerate(indices):
    image, mask = test_dataset[idx]

    with torch.no_grad():
        pred_lit = (torch.sigmoid(model_lit(image.unsqueeze(0).to(device))) > 0.5).float().cpu().squeeze().numpy()
        pred_own_img = (torch.sigmoid(model_own(image.unsqueeze(0).to(device))) > 0.5).float().cpu().squeeze().numpy()

    img_np = image.numpy().transpose(1, 2, 0)
    img_np = (img_np * std + mean).clip(0, 1)
    mask_np = mask.squeeze().numpy()

    d_lit = dice_score(torch.from_numpy(pred_lit).unsqueeze(0), mask.squeeze().unsqueeze(0)).item()
    d_own = dice_score(torch.from_numpy(pred_own_img).unsqueeze(0), mask.squeeze().unsqueeze(0)).item()

    axes[i][0].imshow(img_np)
    axes[i][0].set_title('Imagem Original')
    axes[i][0].axis('off')

    axes[i][1].imshow(mask_np, cmap='gray')
    axes[i][1].set_title('Máscara Real')
    axes[i][1].axis('off')

    axes[i][2].imshow(pred_lit, cmap='gray')
    axes[i][2].set_title(f'Literatura (Dice: {d_lit:.3f})')
    axes[i][2].axis('off')

    axes[i][3].imshow(pred_own_img, cmap='gray')
    axes[i][3].set_title(f'Próprio (Dice: {d_own:.3f})')
    axes[i][3].axis('off')

plt.suptitle('Comparação Visual — Literatura vs Próprio', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Conclusões

**Compilado dos Resultados:**
- Ambos os métodos (Literatura e Próprio) foram capazes de segmentar tumores do tipo Germinoma em imagens de MRI cerebral;
- O método da literatura (U-Net + ResNet34) serve como baseline sólido para a tarefa;
- O método próprio (U-Net + MobileNetV2 + Tversky Loss) oferece uma alternativa com significativamente menos parâmetros, sendo mais eficiente computacionalmente;
- O experimento de redução de dados demonstra a viabilidade de treinar com datasets menores quando combinado com augmentation agressiva.

**Contribuições do Estudo:**
- Análise completa do dataset de tumores cerebrais com foco no Germinoma;
- Implementação e comparação de dois métodos de segmentação;
- Avaliação do impacto da redução de dados no desempenho do modelo;
- Uso de Tversky Loss como alternativa mais adequada para diagnóstico médico.

**Possibilidades de Novos Estudos:**
- Testar arquiteturas com mecanismos de atenção (Attention U-Net);
- Explorar ensemble de modelos para melhorar a robustez;
- Aplicar pós-processamento morfológico nas predições;
- Treinar em múltiplos tipos de tumor e fazer fine-tuning no Germinoma;
- Utilizar técnicas de semi-supervised learning para aproveitar imagens sem anotação.